# Digital Engagement KPIs\n\nBusiness logic:\n1. DAU/MAU from distinct users by day/month.\n2. Login frequency and event intensity by user-day.\n3. Mobile-vs-web usage mix for digital efficiency reporting.

In [ ]:
from pyspark.sql import functions as F\n\nevents = spark.table(\"Digital_EngagementEvents\")\n\nevents_enriched = (\n    events\n    .withColumn(\"EventDate\", F.to_date(\"Timestamp\"))\n    .withColumn(\"EventMonth\", F.date_trunc(\"month\", F.col(\"Timestamp\")))\n)\n\ndaily_users = events_enriched.groupBy(\"EventDate\").agg(F.countDistinct(\"CustomerID\").alias(\"DAU\"))\nmonthly_users = events_enriched.groupBy(\"EventMonth\").agg(F.countDistinct(\"CustomerID\").alias(\"MAU\"))\n\nlogin_frequency = (\n    events_enriched\n    .filter(F.col(\"EventType\") == F.lit(\"Login\"))\n    .groupBy(\"EventDate\")\n    .agg(F.count(\"*\").alias(\"LoginEvents\"), F.countDistinct(\"CustomerID\").alias(\"LoginUsers\"))\n    .withColumn(\"LoginFrequency\", F.round(F.col(\"LoginEvents\") / F.col(\"LoginUsers\"), 3))\n)\n\ndevice_mix = (\n    events_enriched\n    .groupBy(\"EventDate\")\n    .agg(\n        F.count(F.when(F.col(\"Device\") == \"Mobile\", True)).alias(\"MobileEvents\"),\n        F.count(F.when(F.col(\"Device\") == \"Web\", True)).alias(\"WebEvents\"),\n        F.count(\"*\").alias(\"TotalEvents\")\n    )\n    .withColumn(\"MobilePct\", F.round(F.col(\"MobileEvents\") / F.col(\"TotalEvents\"), 4))\n    .withColumn(\"WebPct\", F.round(F.col(\"WebEvents\") / F.col(\"TotalEvents\"), 4))\n)\n\nkpis = (\n    daily_users\n    .join(login_frequency, on=\"EventDate\", how=\"left\")\n    .join(device_mix, on=\"EventDate\", how=\"left\")\n    .withColumn(\"MonthKey\", F.date_trunc(\"month\", F.col(\"EventDate\")))\n    .join(monthly_users.withColumnRenamed(\"EventMonth\", \"MonthKey\"), on=\"MonthKey\", how=\"left\")\n    .drop(\"MonthKey\")\n    .withColumn(\"GeneratedAt\", F.current_timestamp())\n)\n\n(\n    kpis\n    .write.format(\"delta\")\n    .mode(\"overwrite\")\n    .option(\"overwriteSchema\", \"true\")\n    .save(\"Files/Digital/EngagementKPI\")\n)\nspark.sql(\"CREATE TABLE IF NOT EXISTS Digital_EngagementKPI_Daily USING DELTA LOCATION 'Files/Digital/EngagementKPI'\")\n\ndisplay(kpis.orderBy(F.desc(\"EventDate\")).limit(60))\n

## Copilot prompts for narrative generation\n\n-- Generate a client-ready narrative for digital engagement trends.\n-- Describe liquidity changes across buckets.